## Infraestructura utilizada

- Plataforma: Databricks Free Edition.
- Compute: Serverless Starter Warehouse.
- Tamaño: 2X-Small.
- Tipo: Serverless.
- Almacenamiento: Unity Catalog Volumes.
- Ruta del archivo fuente: `/Volumes/workspace/default/data/best-selling-books.csv`.

In [0]:
#Verificamos que este bien el entorno
import sys
print("version Python", sys.version)
print("Versión de Spark:", spark.version)

version Python 3.12.3 (main, Mar 23 2026, 19:04:32) [GCC 13.3.0]
Versión de Spark: 4.1.0


In [0]:
#Definimos el esquema best-selling-books
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

esquema_libros = StructType([
    StructField("Book", StringType(), False),
    StructField("Author(s)", StringType(), True),
    StructField("Original language", StringType(), True),
    StructField("First published", StringType(), True),
    StructField("Approximate sales in millions", DoubleType(), True),
    StructField("Genre", StringType(), True)
])
print("Esquema creado correctamente")


Esquema creado correctamente


## Diccionario de datos

| Campo | Tipo de dato | Llave | Nulos permitidos | Descripción |
|---|---|---|---|---|
| Book | STRING | Sí, llave lógica candidata | No | Nombre del libro |
| Author(s) | STRING | No | Sí | Autor o autores del libro |
| Original language | STRING | No | Sí | Idioma original de publicación |
| First published | STRING | No | Sí | Año o fecha de primera publicación |
| Approximate sales in millions | DOUBLE | No | Sí | Ventas aproximadas en millones |
| Genre | STRING | No | Sí | Género literario |

##Diagrama ER
![image_1780861539146.png](./image_1780861539146.png "image_1780861539146.png")

In [0]:
#Leemos el csv con spark
ruta_csv = "/Volumes/workspace/default/data/best-selling-books.csv"

df_libros = spark.read \
    .option("header", "true") \
    .option("sep", ",") \
    .option("quote", "\"") \
    .option("escape", "\"") \
    .schema(esquema_libros) \
    .csv(ruta_csv)

df_libros.show(10, truncate=False)
df_libros.printSchema()
print("Total de filas:", df_libros.count())
print("Total de columnas:", len(df_libros.columns))

+----------------------------------------+------------------------+-----------------+---------------+-----------------------------+---------------------------+
|Book                                    |Author(s)               |Original language|First published|Approximate sales in millions|Genre                      |
+----------------------------------------+------------------------+-----------------+---------------+-----------------------------+---------------------------+
|A Tale of Two Cities                    |Charles Dickens         |English          |1859           |200.0                        |Historical fiction         |
|The Little Prince (Le Petit Prince)     |Antoine de Saint-Exupéry|French           |1943           |200.0                        |Novella                    |
|Harry Potter and the Philosopher's Stone|J. K. Rowling           |English          |1997           |120.0                        |Fantasy                    |
|And Then There Were None               

In [0]:
#Después de leer el archivo, crea un DataFrame con nombres más cómodos:
from pyspark.sql.functions import col

df_books = df_libros.select(
    col("Book").alias("book"),
    col("Author(s)").alias("author"),
    col("Original language").alias("original_language"),
    col("First published").alias("first_published"),
    col("Approximate sales in millions").alias("approximate_sales_millions"),
    col("Genre").alias("genre")
)

df_books.show(10, truncate=False)
df_books.printSchema()

+----------------------------------------+------------------------+-----------------+---------------+--------------------------+---------------------------+
|book                                    |author                  |original_language|first_published|approximate_sales_millions|genre                      |
+----------------------------------------+------------------------+-----------------+---------------+--------------------------+---------------------------+
|A Tale of Two Cities                    |Charles Dickens         |English          |1859           |200.0                     |Historical fiction         |
|The Little Prince (Le Petit Prince)     |Antoine de Saint-Exupéry|French           |1943           |200.0                     |Novella                    |
|Harry Potter and the Philosopher's Stone|J. K. Rowling           |English          |1997           |120.0                     |Fantasy                    |
|And Then There Were None                |Agatha Christie 

In [0]:
spark.sql("CREATE DATABASE IF NOT EXISTS workspace.default")

# Drop table if exists to avoid schema mismatch
spark.sql("DROP TABLE IF EXISTS workspace.default.best_selling_books")

df_books.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("workspace.default.best_selling_books")

In [0]:
spark.sql("SHOW TABLES IN workspace.default").show(truncate=False)

+--------+------------------+-----------+
|database|tableName         |isTemporary|
+--------+------------------+-----------+
|default |best_selling_books|false      |
+--------+------------------+-----------+



In [0]:
%sql
DESCRIBE TABLE workspace.default.best_selling_books;

col_name,data_type,comment
book,string,null
author,string,null
original_language,string,null
first_published,string,null
approximate_sales_millions,double,null
genre,string,null


In [0]:
%sql
SHOW CREATE TABLE workspace.default.best_selling_books;

createtab_stmt
"CREATE TABLE workspace.default.best_selling_books ( book STRING COLLATE UTF8_BINARY, author STRING COLLATE UTF8_BINARY, original_language STRING COLLATE UTF8_BINARY, first_published STRING COLLATE UTF8_BINARY, approximate_sales_millions DOUBLE, genre STRING COLLATE UTF8_BINARY) USING delta TBLPROPERTIES ( 'delta.enableDeletionVectors' = 'true', 'delta.feature.appendOnly' = 'supported', 'delta.feature.deletionVectors' = 'supported', 'delta.feature.invariants' = 'supported', 'delta.minReaderVersion' = '3', 'delta.minWriterVersion' = '7', 'delta.parquet.compression.codec' = 'zstd')"


In [0]:
%sql
-- Conteo 
SELECT COUNT(*) AS total_registros
FROM workspace.default.best_selling_books;

total_registros
174


In [0]:
%sql
-- Muestra
SELECT * FROM workspace.default.best_selling_books
LIMIT 10;

book,author,original_language,first_published,approximate_sales_millions,genre
A Tale of Two Cities,Charles Dickens,English,1859,200.0,Historical fiction
The Little Prince (Le Petit Prince),Antoine de Saint-Exupéry,French,1943,200.0,Novella
Harry Potter and the Philosopher's Stone,J. K. Rowling,English,1997,120.0,Fantasy
And Then There Were None,Agatha Christie,English,1939,100.0,Mystery
Dream of the Red Chamber (紅樓夢),Cao Xueqin,Chinese,1791,100.0,Family saga
The Hobbit,J. R. R. Tolkien,English,1937,100.0,Fantasy
"The Lion, the Witch and the Wardrobe",C. S. Lewis,English,1950,85.0,"Fantasy, Children's fiction"
She: A History of Adventure,H. Rider Haggard,English,1887,83.0,Adventure
Vardi Wala Gunda (वर्दी वाला गुंडा),Ved Prakash Sharma,Hindi,1992,80.0,Detective
The Da Vinci Code,Dan Brown,English,2003,80.0,Mystery thriller


In [0]:
%sql
-- Agrupacion por idioma
SELECT original_language, COUNT(*) AS cantidad_libros
FROM workspace.default.best_selling_books
GROUP BY original_language
ORDER BY cantidad_libros DESC;

original_language,cantidad_libros
English,131
Russian,6
Japanese,5
French,5
German,5
Chinese,4
Italian,4
Spanish,3
Norwegian,2
Hindi,2


In [0]:
%sql
-- Agrupacion por genero 
SELECT genre, COUNT(*) AS cantidad_libros
FROM workspace.default.best_selling_books
GROUP BY genre
ORDER BY cantidad_libros DESC;

genre,cantidad_libros
null,56
Fantasy,10
Novel,8
Self-help,7
Historical fiction,4
Children's Literature,4
Fiction,3
Thriller,2
"Fantasy, Children's fiction",2
"Children's Literature, picture book",2


In [0]:
%sql
-- Ventas promedio por idioma
SELECT 
    original_language,
    ROUND(AVG(approximate_sales_millions), 2) AS ventas_promedio_millones
FROM workspace.default.best_selling_books
GROUP BY original_language
ORDER BY ventas_promedio_millones DESC;

original_language,ventas_promedio_millones
Portuguese,65.0
French,51.4
Hindi,50.0
Dutch,35.0
Chinese,35.0
English,30.27
Norwegian,30.0
Italian,27.63
Russian,27.23
Spanish,25.0


In [0]:
%sql
-- Filtro
SELECT * FROM workspace.default.best_selling_books
WHERE approximate_sales_millions >= 100
ORDER BY approximate_sales_millions DESC;

book,author,original_language,first_published,approximate_sales_millions,genre
A Tale of Two Cities,Charles Dickens,English,1859,200.0,Historical fiction
The Little Prince (Le Petit Prince),Antoine de Saint-Exupéry,French,1943,200.0,Novella
Harry Potter and the Philosopher's Stone,J. K. Rowling,English,1997,120.0,Fantasy
Dream of the Red Chamber (紅樓夢),Cao Xueqin,Chinese,1791,100.0,Family saga
And Then There Were None,Agatha Christie,English,1939,100.0,Mystery
The Hobbit,J. R. R. Tolkien,English,1937,100.0,Fantasy


In [0]:
# Conteo
df_books.count()

174

In [0]:
# Muestra
df_books.describe().show()

+-------+--------------------+---------------+-----------------+------------------+--------------------------+--------------------+
|summary|                book|         author|original_language|   first_published|approximate_sales_millions|               genre|
+-------+--------------------+---------------+-----------------+------------------+--------------------------+--------------------+
|  count|                 174|            174|              174|               174|                       174|                 118|
|   mean|                NULL|           NULL|             NULL|1962.5229885057472|        30.097126436781608|                NULL|
| stddev|                NULL|           NULL|             NULL| 64.26873708685858|        27.957985464487177|                NULL|
|    min|A Brief History o...|Agatha Christie|          Chinese|              1304|                      10.0|           Adventure|
|    max|Your Erroneous Zones|         Yu Dan|          Yiddish|            

In [0]:
# Agrupacion por idioma
df_books.groupBy("original_language") \
    .count() \
    .orderBy("count", ascending=False) \
    .show(truncate=False)

+-----------------+-----+
|original_language|count|
+-----------------+-----+
|English          |131  |
|Russian          |6    |
|Japanese         |5    |
|German           |5    |
|French           |5    |
|Italian          |4    |
|Chinese          |4    |
|Spanish          |3    |
|Norwegian        |2    |
|Swedish          |2    |
|Hindi            |2    |
|Czech            |1    |
|Yiddish          |1    |
|Dutch            |1    |
|Gujarati         |1    |
|Portuguese       |1    |
+-----------------+-----+



In [0]:
# Agrupacion por genero
df_books.groupBy("genre") \
    .count() \
    .orderBy("count", ascending=False) \
    .show(truncate=False)

+------------------------------------------------------------------------+-----+
|genre                                                                   |count|
+------------------------------------------------------------------------+-----+
|NULL                                                                    |56   |
|Fantasy                                                                 |10   |
|Novel                                                                   |8    |
|Self-help                                                               |7    |
|Historical fiction                                                      |4    |
|Children's Literature                                                   |4    |
|Fiction                                                                 |3    |
|Fantasy, Children's fiction                                             |2    |
|Children's literature                                                   |2    |
|Children's Literature, pict

In [0]:
# Ventas promedio por idioma
from pyspark.sql.functions import avg, round

df_books.groupBy("original_language") \
    .agg(round(avg("approximate_sales_millions"), 2).alias("ventas_promedio_millones")) \
    .orderBy("ventas_promedio_millones", ascending=False) \
    .show(truncate=False)

+-----------------+------------------------+
|original_language|ventas_promedio_millones|
+-----------------+------------------------+
|Portuguese       |65.0                    |
|French           |51.4                    |
|Hindi            |50.0                    |
|Chinese          |35.0                    |
|Dutch            |35.0                    |
|English          |30.27                   |
|Norwegian        |30.0                    |
|Italian          |27.63                   |
|Russian          |27.23                   |
|Spanish          |25.0                    |
|German           |22.6                    |
|Czech            |20.0                    |
|Swedish          |20.0                    |
|Japanese         |14.4                    |
|Yiddish          |10.0                    |
|Gujarati         |10.0                    |
+-----------------+------------------------+



In [0]:
# Filtro
df_books.filter(col("approximate_sales_millions") >= 100) \
    .orderBy(col("approximate_sales_millions").desc()) \
    .show(truncate=False)

+----------------------------------------+------------------------+-----------------+---------------+--------------------------+------------------+
|book                                    |author                  |original_language|first_published|approximate_sales_millions|genre             |
+----------------------------------------+------------------------+-----------------+---------------+--------------------------+------------------+
|A Tale of Two Cities                    |Charles Dickens         |English          |1859           |200.0                     |Historical fiction|
|The Little Prince (Le Petit Prince)     |Antoine de Saint-Exupéry|French           |1943           |200.0                     |Novella           |
|Harry Potter and the Philosopher's Stone|J. K. Rowling           |English          |1997           |120.0                     |Fantasy           |
|And Then There Were None                |Agatha Christie         |English          |1939           |100.0      

In [0]:
# Revision de nulos
from pyspark.sql.functions import count, when, isnan

df_books.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df_books.columns
]).show()

+----+------+-----------------+---------------+--------------------------+-----+
|book|author|original_language|first_published|approximate_sales_millions|genre|
+----+------+-----------------+---------------+--------------------------+-----+
|   0|     0|                0|              0|                         0|   56|
+----+------+-----------------+---------------+--------------------------+-----+



In [0]:
# Revision de duplicados
df_books.groupBy("book") \
    .count() \
    .filter(col("count") > 1) \
    .show(truncate=False)

+----+-----+
|book|count|
+----+-----+
+----+-----+



Estas validaciones permiten identificar campos incompletos, posibles duplicados y verificar la calidad básica del dataset antes de realizar análisis.

## Comparativa SQL vs PySpark

| Criterio | SQL | PySpark |
|---|---|---|
| Facilidad de uso | Más simple para consultas directas | Requiere conocer la API de DataFrames |
| Expresividad | Muy claro para SELECT, JOIN, GROUP BY | Más flexible para transformaciones complejas |
| Integración | Ideal para BI y usuarios analíticos | Mejor para pipelines, ML y procesamiento distribuido |
| Escalabilidad | Escala sobre Spark SQL | Escala usando todo el motor distribuido de Spark |
| Limitaciones | Menos cómodo para lógica compleja o UDFs | Mayor curva de aprendizaje y más código |